In [92]:
from datasets import load_dataset

# Load full dataset
dataset = load_dataset("roneneldan/TinyStories", split="train")

# Sample 1% of the dataset
small_dataset = dataset.train_test_split(test_size=0.1, seed=42)["test"]

print(small_dataset)


Dataset({
    features: ['text'],
    num_rows: 211972
})


In [93]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 doesn't have a padding token, use EOS instead

In [94]:
from transformers import DataCollatorForLanguageModeling, DataCollatorWithPadding

def tokenize_function(element):
    return tokenizer(element["text"], truncation=True, padding="max_length", max_length=256)

# Tokenize dataset
tokenized_dataset = small_dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)

# Create a data collator that dynamically pads batches
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


Map:   0%|          | 0/211972 [00:00<?, ? examples/s]

In [95]:
from transformers import GPT2LMHeadModel, TrainingArguments, Trainer

# Load model
model = GPT2LMHeadModel.from_pretrained('gpt2')

training_args = TrainingArguments(
    output_dir='./checkpoint',  
    num_train_epochs=3,             
    learning_rate=5e-4,
    per_device_train_batch_size=16, 
    gradient_accumulation_steps=4,  
    warmup_steps=200,               
    weight_decay=0.01,              
    logging_first_step=True,        
    
    logging_dir="./checkpoint/logs",
    logging_steps=100,              
    logging_strategy="steps",
    
    report_to="tensorboard",
    
    save_steps=100,                 
    save_total_limit=2,             

    seed=42,                        
    fp16=True,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# Train the model
trainer.train()

# Save final model and tokenizer
model.save_pretrained("./checkpoint")
tokenizer.save_pretrained("./checkpoint")

Step,Training Loss
1,2.647400
100,2.064100
200,1.836000
300,1.760700
400,1.697600
500,1.666600
600,1.645100
700,1.622500
800,1.605100
900,1.580300


('./checkpoint\\tokenizer_config.json',
 './checkpoint\\special_tokens_map.json',
 './checkpoint\\vocab.json',
 './checkpoint\\merges.txt',
 './checkpoint\\added_tokens.json')

In [97]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

def generate_text(prompt, max_length=256, num_return_sequences=1, temperature=0.7, top_k=50, top_p=0.9, repetition_penalty=1.5, no_repeat_ngram_size=2):
    # Load the trained model and tokenizer
    model = GPT2LMHeadModel.from_pretrained("./checkpoint")
    tokenizer = GPT2Tokenizer.from_pretrained("./checkpoint")
    tokenizer.pad_token = tokenizer.eos_token
    
    model.eval()
    
    # Tokenize input prompt
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    attention_mask = input_ids.ne(tokenizer.pad_token_id).long()  # Mask padding tokens
    
    # Generate text with added controls to avoid repetition
    output = model.generate(
        input_ids,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        do_sample=True,  # Enable sampling for more diverse outputs
        attention_mask=attention_mask,  # Provide the attention mask
        pad_token_id=tokenizer.pad_token_id,  # Explicitly set pad_token_id
        eos_token_id=tokenizer.eos_token_id,  # Stop generation at EOS token
        repetition_penalty=repetition_penalty,  # Apply repetition penalty
        no_repeat_ngram_size=no_repeat_ngram_size,  # Avoid repeating n-grams
    )
    
    # Decode and return generated text
    return [tokenizer.decode(seq, skip_special_tokens=True) for seq in output]

# Example usage
prompt = "Once upon a time"
generated_texts = generate_text(prompt)
for i, text in enumerate(generated_texts):
    print(f"Generated Text {i+1}:\n{text}\n")


Generated Text 1:
Once upon a time, there was an enormous bird. It loved to fly high in the sky and look down on its friends below it. One day as it flew higher up into the clouds above them all of sudden something strange happened - The bird noticed that the ground around itself started shaking. Suddenly everything became very heavy!
The bird tried hard not to worry but suddenly things got worse. Big rocks rolled down from the top of the mountain and crashed onto the bottom. All at once the giant rock split apart with great force and shattered into tiny pieces. 


Suddenly the enormous tree nearby shook and branches started swaying and some leaves began falling everywhere. Everyone who saw what had happened rushed out to help. They worked together until they made sure nothing bad would happen again. After this everyone cheered loudly in celebration of their accomplishment. And that's how one brave little girl saved her home from destruction by earthquake-sized rock splitting! 
And so 